# EMRI Step 1 — Geodesic Exploration Notebook

Schwarzschild spacetime · G = c = M = 1 · Equatorial plane

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emri_step1 import (
    SchwarzschildHamiltonian,
    GeodesicSolver,
    EffectivePotential,
    CircularOrbitAnalyzer,
    SolverConfig,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Symbolic Hamiltonian and Equations of Motion

In [ ]:
ham = SchwarzschildHamiltonian()
ham.print_summary()

## 2. Circular Orbit Family

In [ ]:
analyzer = CircularOrbitAnalyzer()
r_arr, E_arr, L_arr = analyzer.isco_scan(r_range=(3.01, 50), n=300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(r_arr, E_arr, 'b-')
axes[0].axvline(6, color='orange', ls='--', label='ISCO r=6')
axes[0].set_xlabel('r / M'); axes[0].set_ylabel('E(r)'); axes[0].set_title('Circular orbit energy')
axes[0].legend()

axes[1].plot(r_arr, L_arr, 'r-')
axes[1].axvline(6, color='orange', ls='--', label='ISCO r=6')
axes[1].set_xlabel('r / M'); axes[1].set_ylabel('L(r)'); axes[1].set_title('Circular orbit angular momentum')
axes[1].legend()
plt.tight_layout()

## 3. Effective Potential

In [ ]:
L_vals = [2.0, 2*np.sqrt(3), 4.0, 5.0]  # 2√3 ≈ ISCO
r_arr = np.linspace(2.1, 25, 500)

fig, ax = plt.subplots(figsize=(9, 5))
for L in L_vals:
    veff = EffectivePotential(L)
    ax.plot(r_arr, veff(r_arr), label=f'L={L:.3f}')

ax.axvline(2, color='black', label='Horizon r=2')
ax.axvline(6, color='orange', ls='--', label='ISCO r=6')
ax.axhline(1.0, color='gray', ls=':', label='E=1 (marginally bound)')
ax.set_xlim(2, 25); ax.set_ylim(0, 1.5)
ax.set_xlabel('r / M'); ax.set_ylabel('V_eff(r; L)')
ax.set_title('Schwarzschild Effective Potential')
ax.legend()
plt.tight_layout()

## 4. Circular Orbit Integration

In [ ]:
solver = GeodesicSolver(ham)
r0 = 10.0
E, L, y0 = solver.ic_from_circular(r0)
print(f'r0={r0}, E={E:.8f}, L={L:.8f}')

cfg = SolverConfig(tau_max=500, n_output=5000)
result = solver.run(E, L, y0, cfg)

print(f'H max drift: {result.H_max_drift:.2e}')
print(f'r range: [{result.r_min:.6f}, {result.r_max:.6f}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Orbit
theta = np.linspace(0, 2*np.pi, 300)
axes[0].plot(result.x, result.y_cart, 'b-', lw=0.8)
axes[0].fill(2*np.cos(theta), 2*np.sin(theta), 'k', alpha=0.2, label='Horizon')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x/M'); axes[0].set_ylabel('y/M'); axes[0].set_title('Orbit')

# r(τ)
axes[1].plot(result.tau, result.r)
axes[1].set_xlabel('τ/M'); axes[1].set_ylabel('r/M'); axes[1].set_title('r(τ)')

# H drift
axes[2].plot(result.tau, result.H_residual, 'r-', lw=0.8)
axes[2].axhline(0, 'k--')
axes[2].set_xlabel('τ/M'); axes[2].set_ylabel('ΔH'); axes[2].set_title('Hamiltonian drift')
axes[2].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

plt.tight_layout()

## 5. Eccentric Orbit

In [ ]:
r_peri, r_apo = 7.0, 20.0
E_ecc, L_ecc, y0_ecc = solver.ic_from_turning_points(r_peri, r_apo)
print(f'E={E_ecc:.6f}, L={L_ecc:.6f}')

cfg_ecc = SolverConfig(tau_max=2000, n_output=15000)
res_ecc = solver.run(E_ecc, L_ecc, y0_ecc, cfg_ecc)

T_r = res_ecc.estimated_radial_period
print(f'Radial period τ_r ≈ {T_r:.2f} M')
print(f'H max drift: {res_ecc.H_max_drift:.2e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta = np.linspace(0, 2*np.pi, 300)
axes[0].plot(res_ecc.x, res_ecc.y_cart, 'b-', lw=0.5)
axes[0].fill(2*np.cos(theta), 2*np.sin(theta), 'k', alpha=0.2, label='Horizon')
axes[0].plot(r_apo, 0, 'go', ms=8, label=f'r_apo={r_apo}')
axes[0].plot(r_peri, 0, 'rs', ms=8, label=f'r_peri={r_peri}')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x/M'); axes[0].set_ylabel('y/M'); axes[0].set_title('Eccentric orbit')
axes[0].legend()

axes[1].plot(res_ecc.tau, res_ecc.r, 'b-', lw=0.8)
axes[1].axhline(r_peri, color='r', ls='--', label=f'r_peri={r_peri}')
axes[1].axhline(r_apo, color='g', ls='--', label=f'r_apo={r_apo}')
axes[1].set_xlabel('τ/M'); axes[1].set_ylabel('r/M'); axes[1].set_title('r(τ) eccentric')
axes[1].legend()
plt.tight_layout()